In [10]:
print("hello ai")

hello ai


In [11]:
import zipfile
import pandas as pd

with zipfile.ZipFile("archive.zip", "r") as z:
    z.extractall(".")

movies = pd.read_csv("tmdb_5000_movies.csv")
print("Total movies:", len(movies))
movies[["title", "genres", "overview"]].head()

Total movies: 4803


,title,genres,overview
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","Captain Barbossa, long believed to be dead, ha..."
2,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",Following the death of District Attorney Harve...
4,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","John Carter is a war-weary, former military ca..."


In [12]:
import ast

def extract_names(text):
    return [item["name"] for item in ast.literal_eval(text)]

movies["genres_clean"] = movies["genres"].apply(extract_names)
movies[["title", "genres_clean"]].head()

,title,genres_clean
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]"
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]"
2,Spectre,"[Action, Adventure, Crime]"
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]"
4,John Carter,"[Action, Adventure, Science Fiction]"


In [13]:
movies["overview"] = movies["overview"].fillna("")
movies["tags"] = (
    movies["overview"] + " " + movies["genres_clean"].apply(lambda x: " ".join(x))
)
movies[["title", "tags"]].head(3)

,title,tags
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
vectors = tfidf.fit_transform(movies["tags"])
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)

Similarity matrix shape: (4803, 4803)


In [15]:
def recommend(movie_title):
    matches = movies[movies["title"].str.lower() == movie_title.lower()]
    if matches.empty:
        print("Movie kandilla! Vere peru try cheyyuka.")
        return
    idx = matches.index[0]
    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    print(f"Top recommendations for '{movies.loc[idx, 'title']}':")
    for i, score in scores:
        print(f"- {movies.loc[i, 'title']} ({score:.2f})")


recommend("Avatar")

Top recommendations for 'Avatar':
- Apollo 18 (0.25)
- Beowulf (0.21)
- The Helix... Loaded (0.20)
- The American (0.20)
- The Adventures of Pluto Nash (0.17)


In [16]:
recommend("The Dark Knight Rises")
recommend("Spectre")

Top recommendations for 'The Dark Knight Rises':
- Batman Forever (0.39)
- The Dark Knight (0.38)
- Batman (0.36)
- Batman Returns (0.32)
- Slow Burn (0.23)
Top recommendations for 'Spectre':
- Never Say Never Again (0.32)
- Thunderball (0.29)
- Quantum of Solace (0.26)
- From Russia with Love (0.26)
- Safe Haven (0.23)


In [17]:
credits = pd.read_csv("tmdb_5000_credits.csv")
movies_full = movies.merge(credits, on="title")


def get_top_cast(text):
    return [
        item["name"].replace(" ", "")
        for item in ast.literal_eval(text)[:3]
    ]


def get_director(text):
    for item in ast.literal_eval(text):
        if item.get("job") == "Director":
            return [item["name"].replace(" ", "")]
    return []


movies_full["cast_clean"] = movies_full["cast"].apply(get_top_cast)
movies_full["director_clean"] = movies_full["crew"].apply(get_director)

movies_full["tags_rich"] = (
    movies_full["overview"]
    + " "
    + movies_full["genres_clean"].apply(lambda x: " ".join(x))
    + " "
    + movies_full["cast_clean"].apply(lambda x: " ".join(x))
    + " "
    + movies_full["director_clean"].apply(lambda x: " ".join(x))
)

tfidf_rich = TfidfVectorizer(max_features=5000, stop_words="english")
vectors_rich = tfidf_rich.fit_transform(movies_full["tags_rich"])
similarity_rich = cosine_similarity(vectors_rich)

print("Upgraded similarity shape:", similarity_rich.shape)

Upgraded similarity shape: (4809, 4809)


In [18]:
def recommend_rich(movie_title):
    matches = movies_full[
        movies_full["title"].str.lower() == movie_title.lower()
    ]
    if matches.empty:
        print("Movie kandilla! Vere peru try cheyyuka.")
        return
    idx = matches.index[0]
    scores = list(enumerate(similarity_rich[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    print(f"Upgraded recommendations for '{movies_full.loc[idx, 'title']}':")
    for i, score in scores:
        print(f"- {movies_full.loc[i, 'title']} ({score:.2f})")


recommend_rich("Avatar")
recommend_rich("The Dark Knight Rises")

Upgraded recommendations for 'Avatar':
- Apollo 18 (0.23)
- The Book of Life (0.22)
- The Helix... Loaded (0.18)
- Aliens (0.18)
- The American (0.17)
Upgraded recommendations for 'The Dark Knight Rises':
- Batman (0.36)
- The Dark Knight (0.36)
- Batman Forever (0.35)
- Batman Returns (0.33)
- Batman Begins (0.31)


In [19]:
import gradio as gr


def get_recommendations_ui(movie_title):
    if not movie_title:
        return "Oru movie select cheyyuka!"
    matches = movies_full[
        movies_full["title"].str.lower() == movie_title.lower()
    ]
    if matches.empty:
        return "Movie kandilla!"
    idx = matches.index[0]
    scores = list(enumerate(similarity_rich[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]
    results = []
    for rank, (i, score) in enumerate(scores, start=1):
        title = movies_full.loc[i, "title"]
        genres = ", ".join(movies_full.loc[i, "genres_clean"])
        results.append(
            f"{rank}. {title}  |  Match: {score:.0%}  |  Genres: {genres}"
        )
    return "\n".join(results)


movie_list = sorted(movies_full["title"].dropna().unique().tolist())

demo = gr.Interface(
    fn=get_recommendations_ui,
    inputs=gr.Dropdown(
        choices=movie_list, label="Select or Type a Movie", value="Avatar"
    ),
    outputs=gr.Textbox(label="Top 5 Recommended Movies", lines=6),
    title="Movie Recommendation System",
    description="Content-Based Filtering using TF-IDF & Cosine Similarity (TMDB 5000 Dataset)",
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://462f1ea0eb71289bd6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
